# Recurrent Network Without an Embedding Layer: One-Hot Inputs

Open in [Google Colab](https://colab.research.google.com/github/febse/ta/blob/main/class/03-14-RNN-Sentiment-One-Hot-Input.ipynb)

In the [previous chapter](03-13-RNN-Sentiment-Classification.ipynb) every word index was first mapped to a vector of 64 numbers by an **embedding layer** and only then passed to the recurrent layer. Here we remove the embedding layer and pass the **one-hot vector** of every word directly to the recurrent layer. The questions are:

- How many parameters does each variant have, and where are they?
- Does the embedding layer improve the predictions, or is it only a way to save parameters?

The data, the preprocessing and the notation are the same as in the previous chapter. We use max pooling over time (see the previous chapter) for the review summary. The numbers quoted in the comments below were obtained by running this code once with the seeds given in the code; they may differ slightly on another machine.

In [1]:
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import nltk
import torch
import torch.nn as nn

## Data

We repeat the preprocessing of the previous chapter without further comments: reviews of unlocked mobile phones, neutral reviews removed, negations kept, stop words removed, Porter stemming, at most 100 words per review, `<pad>` has index 0 and `<unk>` index 1.

In [2]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split

nltk.download("stopwords", quiet=True)

reviews = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/reviews.csv")
reviews = reviews.dropna(subset=["Reviews", "Rating"])
reviews = reviews[reviews["Rating"] != 3]

review_texts = reviews["Reviews"].astype(str).tolist()
sentiment = (reviews["Rating"] > 3).astype(int).to_numpy()

print("reviews:", len(sentiment), " share of positive reviews:", sentiment.mean().round(3))
print(reviews["Rating"].value_counts().sort_index().to_dict())

reviews: 45831  share of positive reviews: 0.745
{1: 8731, 2: 2968, 4: 7297, 5: 26835}


In [3]:
NEGATIONS = {"no", "nor", "not"}          # the negations in the stop list of nltk
STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
stem_cache = {}

def stem(word):
    if word not in stem_cache:
        stem_cache[word] = stemmer.stem(word)
    return stem_cache[word]

def preprocess(text, keep_negations=True):
    text = text.lower().replace("won't", "will not").replace("can't", "can not").replace("n't", " not")
    words = re.findall(r"[a-z]+", text)
    kept = [w for w in words if w not in STOP_WORDS or (keep_negations and w in NEGATIONS)]
    return [stem(w) for w in kept]

print(review_texts[1][:150])
print(preprocess(review_texts[1])[:25])
print(preprocess("The battery isn't good and I can't recommend it, no value for money"))

The phone needed a SIM card, would have been nice to know.
['phone', 'need', 'sim', 'card', 'would', 'nice', 'know']
['batteri', 'not', 'good', 'not', 'recommend', 'no', 'valu', 'money']


In [4]:
MAX_WORDS = 100
MAX_VOCAB = 30000

def build_dataset(keep_negations=True, seed=0):
    docs = [preprocess(text, keep_negations) for text in review_texts]
    idx_tr, idx_rest = train_test_split(np.arange(len(docs)), test_size=0.3, random_state=seed, stratify=sentiment)
    idx_va, idx_te = train_test_split(idx_rest, test_size=0.5, random_state=seed, stratify=sentiment[idx_rest])

    stem_counts = Counter(w for i in idx_tr for w in docs[i])
    stems = ["<pad>", "<unk>"] + [w for w, _ in stem_counts.most_common(MAX_VOCAB)]
    stem_to_id = {w: i for i, w in enumerate(stems)}

    matrix = np.zeros((len(docs), MAX_WORDS), dtype=np.int64)
    lengths = np.zeros(len(docs), dtype=np.int64)
    for i, doc in enumerate(docs):
        encoded = [stem_to_id.get(w, 1) for w in doc][:MAX_WORDS] or [1]    # an empty review becomes one <unk>
        matrix[i, :len(encoded)] = encoded
        lengths[i] = len(encoded)
    return {"docs": docs, "stems": stems, "stem_to_id": stem_to_id, "X": torch.tensor(matrix),
            "lengths": torch.tensor(lengths), "y": torch.tensor(sentiment, dtype=torch.float32),
            "train": idx_tr, "val": idx_va, "test": idx_te, "n_distinct": len(stem_counts)}

review_data = build_dataset()
doc_lengths = np.array([len(d) for d in review_data["docs"]])

print("distinct stems in the training reviews:", review_data["n_distinct"], "-> vocabulary size (with <pad>, <unk>):", len(review_data["stems"]))
print("most common stems:", review_data["stems"][2:17])
print(f"median length: {np.median(doc_lengths):.0f} words, reviews cut at {MAX_WORDS} words: {(doc_lengths > MAX_WORDS).mean():.1%}")
print("train / val / test:", len(review_data["train"]), len(review_data["val"]), len(review_data["test"]))

distinct stems in the training reviews: 15068 -> vocabulary size (with <pad>, <unk>): 15070
most common stems: ['phone', 'not', 'work', 'great', 'use', 'good', 'like', 'one', 'no', 'love', 'get', 'screen', 'batteri', 'would', 'time']
median length: 9 words, reviews cut at 100 words: 3.3%
train / val / test: 32081 6875 6875


## Two Ways to Feed a Word to the Recurrent Layer

Let $V$ be the vocabulary size, $H$ the size of the hidden state and $E$ the size of the embedding. The word $w_t$ at position $t$ has the one-hot vector $x_t \in \{0, 1\}^V$ with a single 1 at the index of the word.

**With an embedding layer** the word is first mapped to $e_t = x_t W_e$ ($W_e$ is $V \times E$) and the recurrent cell reads $e_t$:

:::{.ex-class}
$$
h_t = \tanh\left( \underbrace{x_t W_e}_{e_t} W_x + h_{t-1} W_h + b_h \right), \qquad W_x \in \mathbb{R}^{E \times H}.
$$
:::

**Without an embedding layer** the recurrent cell reads the one-hot vector directly:

:::{.ex-class}
$$
h_t = \tanh\left( x_t W_x + h_{t-1} W_h + b_h \right), \qquad W_x \in \mathbb{R}^{V \times H}.
$$
:::

Because $x_t$ has a single 1, the product $x_t W_x$ simply **selects one row** of $W_x$. So the second network is itself a lookup table, with one learned $H$-dimensional vector per word. The first network learns the same kind of table, but with the factorization $W_e W_x$ of the $V \times H$ matrix through the narrow dimension $E$. The embedding therefore cannot represent anything that the one-hot network cannot (the matrix $W_e W_x$ has rank at most $\min(E, H)$); what it changes is the number of parameters (when $E < H$) and the way they are learned (the product of two matrices is trained, not one matrix).

### Number of Parameters

For a simple RNN the parameters are the matrices above, the bias $b_h$ (PyTorch has two bias vectors in the recurrent layer, $b_{ih}$ and $b_{hh}$, so $2H$ in total) and the output layer ($H$ weights and one bias):

| | embedding | one-hot |
|:---|:---|:---|
| input to hidden | $VE + EH$ | $VH$ |
| hidden to hidden | $H^2$ | $H^2$ |
| biases | $2H$ | $2H$ |
| output layer | $H + 1$ | $H + 1$ |

A GRU has three sets of the matrices $W_x, W_h, b$ (reset gate, update gate, candidate state), an LSTM four. This makes a big difference for the one-hot network: its input matrix is $V \times 3H$, while with an embedding the large matrix $V \times E$ is **shared** by all gates and only the small $E \times 3H$ matrix is multiplied by three.

In [5]:
# the network of the previous chapter (embedding + recurrent layer + pooling)
class EmbeddingSentimentNet(nn.Module):
    def __init__(self, vocab_size, kind="rnn", pooling="max", embedding_size=64, hidden_size=64):
        super().__init__()
        self.kind, self.pooling = kind, pooling
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        layer = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.recurrent = layer(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, word_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(word_ids), lengths, batch_first=True, enforce_sorted=False)
        packed_states, last_state = self.recurrent(packed)
        if self.pooling == "last":
            if self.kind == "lstm":
                last_state = last_state[0]                  # the LSTM also returns its cell state
            summary = last_state[-1]                        # h_T of the last real word
        else:
            states, _ = nn.utils.rnn.pad_packed_sequence(packed_states, batch_first=True)    # (batch, T_max, hidden)
            is_word = (torch.arange(states.shape[1]) < lengths[:, None]).unsqueeze(2)         # False at the padded positions
            if self.pooling == "max":
                summary = states.masked_fill(~is_word, float("-inf")).max(dim=1).values
            else:                                           # "mean"
                summary = (states * is_word).sum(dim=1) / lengths[:, None]
        return self.output(summary).squeeze(1)              # one logit per review

    def logits_after_each_word(self, word_ids):
        # for ONE review (1, T): the logit that we would get if the review ended after word t
        states, _ = self.recurrent(self.embedding(word_ids))
        return self.output(states[0]).squeeze(1)

In [6]:
class OneHotSentimentNet(nn.Module):
    """Same network as SentimentNet, but without the embedding layer: one-hot vectors enter the recurrent layer."""
    def __init__(self, vocab_size, kind="rnn", pooling="max", hidden_size=64):
        super().__init__()
        self.kind, self.pooling, self.vocab_size = kind, pooling, vocab_size
        layer = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.recurrent = layer(vocab_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, word_ids, lengths):
        word_ids = word_ids[:, :int(lengths.max())]                  # drop the padding that no review in the batch uses
        one_hot = nn.functional.one_hot(word_ids, self.vocab_size).float()
        one_hot[:, :, 0] = 0.0                                       # <pad> is the zero vector
        packed = nn.utils.rnn.pack_padded_sequence(one_hot, lengths, batch_first=True, enforce_sorted=False)
        packed_states, last_state = self.recurrent(packed)
        if self.pooling == "last":
            if self.kind == "lstm":
                last_state = last_state[0]
            summary = last_state[-1]
        else:
            states, _ = nn.utils.rnn.pad_packed_sequence(packed_states, batch_first=True)
            is_word = (torch.arange(states.shape[1]) < lengths[:, None]).unsqueeze(2)
            if self.pooling == "max":
                summary = states.masked_fill(~is_word, float("-inf")).max(dim=1).values
            else:
                summary = (states * is_word).sum(dim=1) / lengths[:, None]
        return self.output(summary).squeeze(1)

In [7]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

vocab_size, hidden_size, embedding_size = len(review_data["stems"]), 64, 64
for kind in ["rnn", "gru"]:
    for name, model in [("embedding", EmbeddingSentimentNet(vocab_size, kind)), ("one-hot", OneHotSentimentNet(vocab_size, kind))]:
        print(f"{kind.upper()} {name}: {count_parameters(model):,} parameters")
        for parameter_name, parameter in model.named_parameters():
            print(f"    {parameter_name:<28} {str(tuple(parameter.shape)):<14} {parameter.numel():>10,}")

RNN embedding: 972,865 parameters
    embedding.weight             (15070, 64)       964,480
    recurrent.weight_ih_l0       (64, 64)            4,096
    recurrent.weight_hh_l0       (64, 64)            4,096
    recurrent.bias_ih_l0         (64,)                  64
    recurrent.bias_hh_l0         (64,)                  64
    output.weight                (1, 64)                64
    output.bias                  (1,)                    1
RNN one-hot: 968,769 parameters
    recurrent.weight_ih_l0       (64, 15070)       964,480
    recurrent.weight_hh_l0       (64, 64)            4,096
    recurrent.bias_ih_l0         (64,)                  64
    recurrent.bias_hh_l0         (64,)                  64
    output.weight                (1, 64)                64
    output.bias                  (1,)                    1
GRU embedding: 989,505 parameters
    embedding.weight             (15070, 64)       964,480
    recurrent.weight_ih_l0       (192, 64)          12,288
    recurrent.w

The counts agree with the formulas: for the simple RNN the one-hot network has $EH = 4{,}096$ parameters **fewer** than the embedding network (the embedding adds the matrix $W_x$ of size $E \times H$ on top of $V \times E$, and here $E = H$). For the GRU the one-hot network has about three times as many parameters (2.9 million against 1.0 million), almost all of them in the input matrix.

The next cell computes the formulas and shows how the counts change with the embedding size $E$ (with $H = 64$ fixed).

In [8]:
def n_parameters(kind, vocab_size, hidden_size, embedding_size=None):
    gates = {"rnn": 1, "gru": 3, "lstm": 4}[kind]
    if embedding_size is None:                                  # one-hot input
        input_part = vocab_size * gates * hidden_size
    else:
        input_part = vocab_size * embedding_size + embedding_size * gates * hidden_size
    recurrent_part = gates * (hidden_size * hidden_size + 2 * hidden_size)
    return input_part + recurrent_part + hidden_size + 1

print(f"{'':<22} {'RNN':>12} {'GRU':>12} {'LSTM':>12}")
for label, e in [("one-hot (no embedding)", None), ("embedding E=128", 128), ("embedding E=64", 64), ("embedding E=32", 32), ("embedding E=16", 16)]:
    print(f"{label:<22}", *[f"{n_parameters(kind, vocab_size, hidden_size, e):>12,}" for kind in ["rnn", "gru", "lstm"]])

# the formula agrees with PyTorch
assert n_parameters("gru", vocab_size, hidden_size, 64) == count_parameters(EmbeddingSentimentNet(vocab_size, "gru"))
assert n_parameters("gru", vocab_size, hidden_size) == count_parameters(OneHotSentimentNet(vocab_size, "gru"))

                                RNN          GRU         LSTM
one-hot (no embedding)      968,769    2,906,177    3,874,881
embedding E=128           1,941,441    1,966,273    1,978,689
embedding E=64              972,865      989,505      997,825
embedding E=32              488,577      501,121      507,393
embedding E=16              246,433      256,929      262,177


### The Embedding Is a Factorized Lookup Table

We check the claim that a one-hot network with the matrix $W_e W_x$ as its input matrix is the same function as the embedding network. We copy the weights of an (untrained) embedding RNN into a one-hot RNN and compare the logits of the same reviews.

In [9]:
# no-ex-class
torch.manual_seed(0)
embedding_net = EmbeddingSentimentNet(vocab_size, "rnn").eval()
one_hot_net = OneHotSentimentNet(vocab_size, "rnn").eval()

with torch.no_grad():
    embedding_matrix = embedding_net.embedding.weight.clone()
    embedding_matrix[0] = 0.0                                                       # <pad> is the zero vector
    one_hot_net.recurrent.weight_ih_l0.copy_((embedding_matrix @ embedding_net.recurrent.weight_ih_l0.T).T)    # (hidden, vocab)
    for name in ["weight_hh_l0", "bias_ih_l0", "bias_hh_l0"]:
        getattr(one_hot_net.recurrent, name).copy_(getattr(embedding_net.recurrent, name))
    one_hot_net.output.load_state_dict(embedding_net.output.state_dict())

    X_check, lengths_check = review_data["X"][:200], review_data["lengths"][:200]
    difference = (embedding_net(X_check, lengths_check) - one_hot_net(X_check, lengths_check)).abs().max()
print("largest difference between the logits:", difference.item())

largest difference between the logits: 5.960464477539063e-08


## Training and Comparison

We train both networks (simple RNN and GRU) with the same settings as in the previous chapter and with max pooling. To keep the running time reasonable we train for 5 epochs (the best validation epoch in the previous chapter was 2 to 4 for max pooling) and use three seeds. The training functions are the same as before. Note that the one-hot network has to build a $B \times T \times V$ array of zeros and ones for every batch ($V \approx 15{,}000$), which is slow and memory-hungry; this is the practical reason why nobody feeds one-hot vectors to a network directly with a large vocabulary.

In [10]:
bce_loss = nn.BCEWithLogitsLoss()

def split_tensors(data, name):
    rows = torch.tensor(data[name])
    return data["X"][rows], data["lengths"][rows], data["y"][rows]

def predict_logits(model, X, lengths, batch_size=512):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[s:s + batch_size], lengths[s:s + batch_size]) for s in range(0, len(X), batch_size)])

def fit_sentiment(model, data, n_epochs=8, learning_rate=2e-3, batch_size=64, clip=5.0, seed=0):
    X_tr, len_tr, y_tr = split_tensors(data, "train")
    X_va, len_va, y_va = split_tensors(data, "val")
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            bce_loss(model(X_tr[batch], len_tr[batch]), y_tr[batch]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        history["val"].append(bce_loss(predict_logits(model, X_va, len_va), y_va).item())
        if history["val"][-1] < best_loss:
            best_loss, best_state, history["best_epoch"] = history["val"][-1], {k: v.clone() for k, v in model.state_dict().items()}, epoch
    model.load_state_dict(best_state)
    return model, history

def test_scores(model, data):
    X_te, len_te, y_te = split_tensors(data, "test")
    probs = torch.sigmoid(predict_logits(model, X_te, len_te)).numpy()
    return probs, y_te.numpy()

In [11]:
import time
from sklearn.metrics import roc_auc_score

comparison = {}
for kind in ["rnn", "gru"]:
    for variant, network in [("embedding", EmbeddingSentimentNet), ("one-hot", OneHotSentimentNet)]:
        runs = []
        for seed in [0, 1, 2]:
            torch.manual_seed(seed)
            model = network(vocab_size, kind)
            start = time.time()
            model, history = fit_sentiment(model, review_data, n_epochs=5, seed=seed)
            seconds_per_epoch = (time.time() - start) / 5
            probs, truth = test_scores(model, review_data)
            runs.append((((probs > 0.5) == truth).mean(), roc_auc_score(truth, probs), history["best_epoch"] + 1, seconds_per_epoch))
        comparison[(kind, variant)] = (count_parameters(model), np.array(runs))

print(f"{'model':<5} {'input':<10} {'parameters':>11} {'test accuracy':>16} {'test AUC':>16} {'best epochs':>12} {'sec/epoch':>10}")
for (kind, variant), (n_params, runs) in comparison.items():
    acc, auc = runs[:, 0], runs[:, 1]
    print(f"{kind.upper():<5} {variant:<10} {n_params:>11,} {acc.mean():>9.3f} ± {acc.std(ddof=1):.3f} {auc.mean():>9.3f} ± {auc.std(ddof=1):.3f} "
          f"{str([int(e) for e in runs[:, 2]]):>12} {runs[:, 3].mean():>10.0f}")

model input       parameters    test accuracy         test AUC  best epochs  sec/epoch
RNN   embedding      972,865     0.939 ± 0.001     0.978 ± 0.001    [4, 4, 3]          5
RNN   one-hot        968,769     0.943 ± 0.002     0.980 ± 0.000    [3, 3, 3]         96
GRU   embedding      989,505     0.944 ± 0.003     0.979 ± 0.001    [3, 3, 3]         14
GRU   one-hot      2,906,177     0.940 ± 0.002     0.979 ± 0.000    [2, 2, 3]        110


### Evaluating the Effect of the Embedding

**Parameters.** In both networks almost all parameters belong to the input matrix: 964,480 of 972,865 (99.1%) in the embedding RNN, and 964,480 of 968,769 in the one-hot RNN; the recurrent part itself has only a few thousand. For the simple RNN the two variants are practically the same size, and the one-hot network is even slightly smaller (968,769 against 972,865, the difference of $EH = 4{,}096$ predicted above). For the GRU the one-hot network is three times larger (2,906,177 against 989,505), because its input matrix is $V \times 3H$. The embedding makes the model size almost independent of the type of recurrent cell, and it is the only one of the two designs whose size can be reduced further by choosing $E < H$ (246,433 parameters for the RNN with $E = 16$). The check with copied weights confirms that the embedding network is the one-hot network with the input matrix $W_e W_x$ (largest logit difference $6 \cdot 10^{-8}$).

**Prediction quality.** Test AUC is the same within noise for all four models (0.978 to 0.980), and the test accuracies lie within 0.4 percentage points (93.9% to 94.4%). The ordering is not consistent: for the simple RNN the one-hot input is slightly better (94.3% against 93.9%, about two standard deviations), for the GRU the embedding is slightly better (94.4% against 94.0%, about one and a half standard deviations). With three seeds we cannot call either difference real. The honest conclusion is that **the embedding does not improve the predictions here**: the one-hot network can learn exactly the same word vectors (it is a lookup table with one vector per word). The gain from the embedding is in size and speed, not in accuracy. A possible reason for the weak advantage of the one-hot variant for the RNN is that it has one fewer matrix product to learn (one matrix against a product of two), which makes the optimization simpler; we did not test this.

**Training time.** The one-hot networks need 96 and 110 seconds per epoch against 5 and 14 seconds for the embedding networks (about 20 and 8 times slower on this machine), because every batch is multiplied as a dense $B \times T \times V$ array. This is a property of the dense implementation: the row lookup that `nn.Embedding` performs gives the same result without building the array.

**What the embedding would change.** This vocabulary is small (15,070 stems) and the data set has 32,000 training reviews, and the sentiment is carried by individual words, so the one-hot lookup table is learned well. The advantages of an embedding that we did not test here are the compact size for $E < H$ and the possibility to start from pre-trained vectors learned on large corpora (see the Word2Vec chapter), which helps most when the labeled data set is small.

## Limitations

- **Large vocabularies.** The one-hot input needs a $B \times T \times V$ array per batch; with a vocabulary of several hundred thousand words this is impossible. In practice the multiplication $x_t W_x$ is implemented as a row lookup, which is exactly what `nn.Embedding` does.
- **Same-size comparison.** We compared one embedding size ($E = H = 64$). With a smaller $E$ the embedding network would have fewer parameters and might predict differently.
- **A few runs.** Three seeds per setting; differences of a few tenths of a percentage point are not meaningful.

## Summary

- A one-hot vector times a weight matrix selects a row of that matrix, so a recurrent layer on one-hot inputs contains a lookup table with one $H$-dimensional vector per word.
- An embedding layer factorizes that table through a narrower dimension $E$. For a simple RNN with $E = H$ it adds $EH$ parameters; for gated networks (GRU, LSTM), where the input matrix is $3H$ or $4H$ wide, it shares the big $V \times E$ matrix between the gates and saves about two thirds of the parameters.
- Dense one-hot inputs are slow and memory-hungry, so embeddings (as lookups) are the practical choice.

## Exercises

1. Compute with `n_parameters` for which embedding size $E$ the GRU with an embedding has the same number of parameters as the simple RNN without one.
2. Train the embedding networks with $E = 16$ and $E = 32$. How do the parameters and the test accuracy change?
3. Replace the recurrent layer by the sum of the one-hot vectors of a review (the bag of words) followed by a logistic regression. How many parameters does it have, and how is it related to the one-hot network with mean pooling?